1. Baca seluruh kolom dalam bentuk text

In [1]:
from pathlib import Path
import polars as pl

DATA_PATH = Path("../data/raw/stocks.csv")

df = pl.read_csv(DATA_PATH, separator=";", infer_schema=False)

print("Shape:", df.shape)
print("Kolom:", df.columns)
print(f"Ukuran di memori: {df.estimated_size('mb'):.1f} MB")
df.head(10)

Shape: (1048575, 7)
Kolom: ['datetime', 'symbol', 'open', 'high', 'low', 'close', 'volume']
Ukuran di memori: 51.7 MB


datetime,symbol,open,high,low,close,volume
str,str,str,str,str,str,str
"""30/06/2005 02:00""","""IDX:AALI""","""311192455""","""3131620275""","""311192455""","""3131620275""","""15809260"""
"""01/07/2005 02:00""","""IDX:AALI""","""311192455""","""311192455""","""30725331""","""3092228825""","""9012100"""
"""04/07/2005 02:00""","""IDX:AALI""","""30725331""","""30725331""","""29937502""","""303314165""","""32265880"""
"""05/07/2005 02:00""","""IDX:AALI""","""303314165""","""303314165""","""3013445925""","""3013445925""","""19287180"""
"""06/07/2005 02:00""","""IDX:AALI""","""303314165""","""30725331""","""303314165""","""3052837375""","""16323330"""
"""07/07/2005 02:00""","""IDX:AALI""","""3052837375""","""311192455""","""3052837375""","""3092228825""","""22727010"""
"""08/07/2005 02:00""","""IDX:AALI""","""3092228825""","""319070745""","""3092228825""","""319070745""","""32538780"""
"""11/07/2005 02:00""","""IDX:AALI""","""319070745""","""32300989""","""3171011725""","""319070745""","""12052100"""
"""12/07/2005 02:00""","""IDX:AALI""","""319070745""","""3210403175""","""3151316""","""3171011725""","""15967930"""


2. Validasi Struktur

In [2]:
import csv
from collections import Counter

with open(DATA_PATH, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f, delimiter=";")
    next(reader)
    counts = Counter(len(row) for row in reader)

print("Distribusi jumlah kolom per baris:", dict(counts))

Distribusi jumlah kolom per baris: {7: 1048575}


3. Cek null dan string kosong

In [3]:
profile_missing = pl.DataFrame({
    "kolom": df.columns,
    "null": [df[c].null_count() for c in df.columns],
    "kosong": [(df[c].str.strip_chars() == "").sum() for c in df.columns],
}).with_columns(
    (pl.col("null") / df.height * 100).round(3).alias("pct_null")
)
profile_missing

kolom,null,kosong,pct_null
str,i64,i64,f64
"""datetime""",0,0,0.0
"""symbol""",0,0,0.0
"""open""",0,0,0.0
"""high""",0,0,0.0
"""low""",0,0,0.0
"""close""",0,0,0.0
"""volume""",0,0,0.0


4. Profile kolom datetime

In [4]:
dt = df.select(
    pl.col("datetime"),
    pl.col("datetime").str.strptime(pl.Datetime, "%d/%m/%Y %H:%M", strict=False).alias("dt_parsed"),
)

gagal = dt.filter(pl.col("dt_parsed").is_null())
print("Gagal parse datetime:", gagal.height)
print(gagal.head(10))

print("Rentang tanggal:", dt["dt_parsed"].min(), "->", dt["dt_parsed"].max())

# apakah jam selalu sama?
print(dt.select(pl.col("dt_parsed").dt.strftime("%H:%M").alias("jam"))
        .group_by("jam").len().sort("len", descending=True).head(5))

Gagal parse datetime: 0
shape: (0, 2)
┌──────────┬──────────────┐
│ datetime ┆ dt_parsed    │
│ ---      ┆ ---          │
│ str      ┆ datetime[μs] │
╞══════════╪══════════════╡
└──────────┴──────────────┘
Rentang tanggal: 1990-05-08 02:00:00 -> 2025-07-04 02:00:00
shape: (1, 2)
┌───────┬─────────┐
│ jam   ┆ len     │
│ ---   ┆ ---     │
│ str   ┆ u32     │
╞═══════╪═════════╡
│ 02:00 ┆ 1048575 │
└───────┴─────────┘


5. Profile kolom simbol

In [5]:
sym = df["symbol"]
print("Jumlah simbol unik:", sym.n_unique())
print("Simbol tanpa prefix 'IDX:':", (~sym.str.starts_with("IDX:")).sum())

per_simbol = (
    df.group_by("symbol").len().sort("len", descending=True)
)
print(per_simbol.head(10))
print(per_simbol.tail(10))
print(per_simbol["len"].describe())

Jumlah simbol unik: 318
Simbol tanpa prefix 'IDX:': 0
shape: (10, 2)
┌──────────┬──────┐
│ symbol   ┆ len  │
│ ---      ┆ ---  │
│ str      ┆ u32  │
╞══════════╪══════╡
│ IDX:GJTL ┆ 8434 │
│ IDX:ASII ┆ 7658 │
│ IDX:JPFA ┆ 7415 │
│ IDX:BBNI ┆ 6913 │
│ IDX:AKRA ┆ 6516 │
│ IDX:BBCA ┆ 6065 │
│ IDX:BMRI ┆ 5318 │
│ IDX:BBRI ┆ 5219 │
│ IDX:GGRM ┆ 4831 │
│ IDX:AALI ┆ 4831 │
└──────────┴──────┘
shape: (10, 2)
┌──────────┬──────┐
│ symbol   ┆ len  │
│ ---      ┆ ---  │
│ str      ┆ u32  │
╞══════════╪══════╡
│ IDX:MGNA ┆ 1388 │
│ IDX:MAGP ┆ 1379 │
│ IDX:IBST ┆ 1315 │
│ IDX:JSPT ┆ 1142 │
│ IDX:CTBN ┆ 1139 │
│ IDX:GOLL ┆ 963  │
│ IDX:BCIC ┆ 847  │
│ IDX:HDTX ┆ 831  │
│ IDX:BSWD ┆ 715  │
│ IDX:BPII ┆ 684  │
└──────────┴──────┘
shape: (9, 2)
┌────────────┬────────────┐
│ statistic  ┆ value      │
│ ---        ┆ ---        │
│ str        ┆ f64        │
╞════════════╪════════════╡
│ count      ┆ 318.0      │
│ null_count ┆ 0.0        │
│ mean       ┆ 3297.40566 │
│ std        ┆ 1144.29561 │
│ min     

6. profil format angka

In [6]:
num_cols = ["open", "high", "low", "close", "volume"]

def pola(c):
    s = pl.col(c)
    return (
        pl.when(s.is_null()).then(pl.lit("null"))
        .when(s.str.contains(r"[Ee][+-]?\d+")).then(pl.lit("notasi_ilmiah"))
        .when(s.str.contains(",")).then(pl.lit("ada_koma"))
        .when(s.str.contains(r"\.")).then(pl.lit("ada_titik"))
        .when(s.str.contains(r"^\d+$")).then(pl.lit("digit_saja"))
        .otherwise(pl.lit("lainnya"))
        .alias(c)
    )

for c in num_cols:
    print(f"\n=== {c} ===")
    print(df.select(pola(c)).group_by(c).len().sort("len", descending=True))

# panjang digit untuk kolom harga, untuk melihat apakah skalanya konsisten
for c in ["open", "close"]:
    print(f"\nDistribusi panjang string: {c}")
    print(df.select(pl.col(c).str.len_chars().alias("panjang"))
            .group_by("panjang").len().sort("panjang"))


=== open ===
shape: (2, 2)
┌───────────────┬─────────┐
│ open          ┆ len     │
│ ---           ┆ ---     │
│ str           ┆ u32     │
╞═══════════════╪═════════╡
│ digit_saja    ┆ 1043883 │
│ notasi_ilmiah ┆ 4692    │
└───────────────┴─────────┘

=== high ===
shape: (2, 2)
┌───────────────┬─────────┐
│ high          ┆ len     │
│ ---           ┆ ---     │
│ str           ┆ u32     │
╞═══════════════╪═════════╡
│ digit_saja    ┆ 1043816 │
│ notasi_ilmiah ┆ 4759    │
└───────────────┴─────────┘

=== low ===
shape: (2, 2)
┌───────────────┬─────────┐
│ low           ┆ len     │
│ ---           ┆ ---     │
│ str           ┆ u32     │
╞═══════════════╪═════════╡
│ digit_saja    ┆ 1043945 │
│ notasi_ilmiah ┆ 4630    │
└───────────────┴─────────┘

=== close ===
shape: (2, 2)
┌───────────────┬─────────┐
│ close         ┆ len     │
│ ---           ┆ ---     │
│ str           ┆ u32     │
╞═══════════════╪═════════╡
│ digit_saja    ┆ 1043893 │
│ notasi_ilmiah ┆ 4682    │
└───────────────┴───

7. Konversi ke angka dan hitung kegagalan

In [7]:
def ke_float(c):
    return (
        pl.col(c).str.strip_chars()
        .str.replace_all(r"\.", "")      # buang titik ribuan (jika ada)
        .str.replace(",", ".")           # koma desimal -> titik
        .cast(pl.Float64, strict=False)
        .alias(c)
    )

clean = df.with_columns(
    pl.col("datetime").str.strptime(pl.Datetime, "%d/%m/%Y %H:%M", strict=False),
    *[ke_float(c) for c in num_cols],
)

gagal_num = pl.DataFrame({
    "kolom": num_cols,
    "gagal_parse": [
        (clean[c].is_null() & df[c].is_not_null()).sum() for c in num_cols
    ],
})
print(gagal_num)
clean.describe()

shape: (5, 2)
┌────────┬─────────────┐
│ kolom  ┆ gagal_parse │
│ ---    ┆ ---         │
│ str    ┆ i64         │
╞════════╪═════════════╡
│ open   ┆ 0           │
│ high   ┆ 0           │
│ low    ┆ 0           │
│ close  ┆ 0           │
│ volume ┆ 0           │
└────────┴─────────────┘


statistic,datetime,symbol,open,high,low,close,volume
str,str,str,f64,f64,f64,f64,f64
"""count""","""1048575""","""1048575""",1.048575e6,1.048575e6,1.048575e6,1.048575e6,1.048575e6
"""null_count""","""0""","""0""",0.0,0.0,0.0,0.0,0.0
"""mean""","""2016-10-29 16:36:59.480724""",null,2.1977e9,2.2284e9,2.1758e9,2.2053e9,1.6363e14
"""std""",null,null,3.0154e10,3.0465e10,2.9701e10,3.0193e10,1.6134e15
"""min""","""1990-05-08 02:00:00""","""IDX:AALI""",10.0,20.0,10.0,10.0,0.0
"""25%""","""2012-09-12 02:00:00""",null,1820.0,1870.0,1780.0,1820.0,482000.0
"""50%""","""2017-07-31 02:00:00""",null,7000.0,7150.0,6900.0,7000.0,1.0345e7
"""75%""","""2021-08-18 02:00:00""",null,74450.0,75750.0,73750.0,74500.0,1.2947e8
"""max""","""2025-07-04 02:00:00""","""IDX:MLBI""",1.1498e12,1.1783e12,1.1261e12,1.1641e12,5.3671e16


8. Cek kualitas dan logika data

In [8]:
print("Duplikat (symbol, datetime):",
      clean.select(["symbol", "datetime"]).is_duplicated().sum())
print("Baris duplikat penuh:", clean.is_duplicated().sum())

checks = {
    "harga <= 0": clean.filter(pl.any_horizontal(pl.col(["open","high","low","close"]) <= 0)).height,
    "volume < 0": clean.filter(pl.col("volume") < 0).height,
    "volume == 0": clean.filter(pl.col("volume") == 0).height,
    "high < low": clean.filter(pl.col("high") < pl.col("low")).height,
    "close di luar [low, high]": clean.filter(
        (pl.col("close") > pl.col("high")) | (pl.col("close") < pl.col("low"))
    ).height,
    "open di luar [low, high]": clean.filter(
        (pl.col("open") > pl.col("high")) | (pl.col("open") < pl.col("low"))
    ).height,
}
for k, v in checks.items():
    print(f"{k:30s}: {v:>10,} ({v / clean.height * 100:.3f}%)")

Duplikat (symbol, datetime): 0
Baris duplikat penuh: 0
harga <= 0                    :          0 (0.000%)
volume < 0                    :          0 (0.000%)
volume == 0                   :     18,114 (1.727%)
high < low                    :     37,876 (3.612%)
close di luar [low, high]     :     58,792 (5.607%)
open di luar [low, high]      :     56,953 (5.431%)


9. Deteksi salah skala

In [9]:
ret = (
    clean.sort(["symbol", "datetime"])
    .with_columns(
        (pl.col("close") / pl.col("close").shift(1).over("symbol")).alias("rasio_close")
    )
)

print("Baris dengan perubahan harian > 10x atau < 0.1x:")
ret.filter((pl.col("rasio_close") > 10) | (pl.col("rasio_close") < 0.1)) \
   .select(["datetime", "symbol", "close", "rasio_close"]).head(20)

Baris dengan perubahan harian > 10x atau < 0.1x:


datetime,symbol,close,rasio_close
datetime[μs],str,f64,f64
2005-07-04 02:00:00,"""IDX:AALI""",3.03314165e8,0.098089
2005-07-13 02:00:00,"""IDX:AALI""",3.151316e6,0.000994
2005-07-14 02:00:00,"""IDX:AALI""",3.11192455e8,98.75
2005-07-21 02:00:00,"""IDX:AALI""",3.2498e9,10.185185
2005-07-22 02:00:00,"""IDX:AALI""",3.2300989e7,0.009939
…,…,…,…
2005-09-05 02:00:00,"""IDX:AALI""",3.2300989e7,0.010314
2005-09-06 02:00:00,"""IDX:AALI""",3.3680e9,104.268293
2005-09-07 02:00:00,"""IDX:AALI""",3.4664476e7,0.010292
